**Installing Dependencies**

In [1]:
# from google.cloud import resourcemanager_v3
# import google.auth

# credentials, project_id = google.auth.default()
# client = resourcemanager_v3.ProjectsClient(credentials=credentials)

# # Retrieve IAM Policy for the active project
# request = resourcemanager_v3.GetIamPolicyRequest(
#     resource=f"projects/{project_id}"
# )
# policy = client.get_iam_policy(request=request)

# # Identify active identity
# principal = credentials.service_account_email if hasattr(credentials, 'service_account_email') else 'your active credential'
# print(f"Searching roles for principal: {principal}\n")

# found = False
# for binding in policy.bindings:
#     # Check if the current credential is in the members list of this role binding
#     members = [m.lower() for m in binding.members]
#     match_terms = [principal.lower()]
#     if hasattr(credentials, 'signer_email') and credentials.signer_email:
#         match_terms.append(credentials.signer_email.lower())

#     if any(any(term in member for term in match_terms) for member in members):
#         print(f"Role: {binding.role}")
#         found = True

# if not found:
#     print("Could not find direct bindings. You might have inherited permissions from folder/organization levels.")

In [2]:
%pip install -q -U "google-cloud-aiplatform[agent_engines,adk]" google-adk google-genai google-cloud-logging google-cloud-secret-manager requests google-auth

**Configure project, model, and credentials**

In [3]:
import os, re, json, uuid, logging, hashlib
from datetime import datetime, timezone
import google.auth
import vertexai

PROJECT_ID = "qwiklabs-gcp-00-e17665342e06"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://seq-query-agent-staging-2026-tiachallenge5"
MODEL = "gemini-2.5-flash"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
credentials, active_project = google.auth.default()
vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials, staging_bucket=STAGING_BUCKET)
print("Vertex AI ready:", PROJECT_ID, LOCATION)
print("Active credential project:", active_project)


Vertex AI ready: qwiklabs-gcp-00-e17665342e06 us-central1
Active credential project: qwiklabs-gcp-00-e17665342e06


**Google Maps credentials**

In [4]:
import requests
from urllib.parse import quote
import google.auth
import google.auth.transport.requests

HTTP = requests.Session()
HTTP.headers.update({"User-Agent": "ReadyNowPOC/1.0 (contact: replace-with-real-contact@example.com)"})

def _get_adc_token() -> str:
    """Retrieve an OAuth 2.0 Access Token via Application Default Credentials (ADC)."""
    try:
        credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
        auth_req = google.auth.transport.requests.Request()
        credentials.refresh(auth_req)
        return credentials.token
    except Exception as exc:
        print("Error acquiring ADC OAuth token:", exc)
        return ""

def geocode_location(location: str) -> dict:
    """Resolve a US city/address into coordinates using Google Maps Geocoding API via ADC OAuth."""
    if not isinstance(location, str) or not (2 <= len(location.strip()) <= 180):
        return {"status": "invalid_input", "message": "Provide a city/state or address."}

    token = _get_adc_token()
    if not token:
        return {"status": "unavailable", "message": "Application Default Credentials (ADC) token is unavailable."}

    try:
        headers = {"Authorization": f"Bearer {token}"}
        r = HTTP.get("https://maps.googleapis.com/maps/api/geocode/json", params={
            "address": location, "components": "country:US"
        }, headers=headers, timeout=15)
        r.raise_for_status()
        data = r.json()
        if data.get("status") != "OK" or not data.get("results"):
            return {"status": "unavailable", "message": "Location could not be resolved", "api_status": data.get("status")}
        result = data["results"][0]
        loc = result["geometry"][0] if isinstance(result["geometry"], list) else result["geometry"]["location"]
        return {"status": "success", "formatted_address": result["formatted_address"],
                "latitude": loc["lat"], "longitude": loc["lng"], "source": "Google Maps Geocoding API via ADC"}
    except requests.RequestException:
        return {"status": "unavailable", "message": "Geocoding service could not be reached."}

** Weather **

In [5]:
NWS_HEADERS = {"Accept": "application/geo+json", "User-Agent": "ReadyNowPOC/1.0 (contact: replace-with-real-contact@example.com)"}

def get_weather_and_alerts(latitude: float, longitude: float) -> dict:
    """Retrieve NWS hourly forecast and active alerts at US coordinates."""
    try:
        lat, lon = float(latitude), float(longitude)
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            return {"status": "invalid_input", "message": "Coordinates out of range."}
        checked = datetime.now(timezone.utc).isoformat()
        alerts_r = HTTP.get("https://api.weather.gov/alerts/active", params={"point": f"{lat},{lon}"}, headers=NWS_HEADERS, timeout=20)
        alerts_r.raise_for_status()
        alerts = [{"event": p.get("event"), "severity": p.get("severity"), "headline": p.get("headline"),
                   "instruction": p.get("instruction"), "description": p.get("description"),
                   "expires": p.get("expires"), "url": p.get("@id") or p.get("id")}
                  for a in alerts_r.json().get("features", []) for p in [a.get("properties", {})]]
        point_r = HTTP.get(f"https://api.weather.gov/points/{lat},{lon}", headers=NWS_HEADERS, timeout=20)
        point_r.raise_for_status()
        forecast_url = point_r.json()["properties"]["forecastHourly"]
        forecast_r = HTTP.get(forecast_url, headers=NWS_HEADERS, timeout=20)
        forecast_r.raise_for_status()
        periods = forecast_r.json()["properties"].get("periods", [])[:6]
        forecast = [{"time": p.get("startTime"), "temperature": p.get("temperature"),
                     "unit": p.get("temperatureUnit"), "conditions": p.get("shortForecast"),
                     "wind": p.get("windSpeed"), "precipitation_probability":
                     (p.get("probabilityOfPrecipitation") or {}).get("value")}
                    for p in periods]
        return {"status": "success", "checked_at_utc": checked, "latitude": lat, "longitude": lon,
                "alerts": alerts, "forecast": forecast, "source": "https://www.weather.gov/"}
    except (requests.RequestException, KeyError, ValueError, TypeError) as exc:
        return {"status": "unavailable", "message": "Could not verify current NWS data.", "error_type": type(exc).__name__}


**Google Maps Routes API**

In [6]:
def get_google_maps_route(origin: str, destination: str) -> dict:
    """Calculate traffic-aware driving directions via Routes API using ADC OAuth; NOT a verified evacuation route."""
    if not isinstance(origin, str) or not isinstance(destination, str) or not origin.strip() or not destination.strip():
        return {"status": "invalid_input", "message": "Provide origin and a verified destination."}
    if len(origin) > 180 or len(destination) > 180:
        return {"status": "invalid_input", "message": "Location text is too long."}

    token = _get_adc_token()
    if not token:
        return {"status": "unavailable", "message": "Application Default Credentials (ADC) token is unavailable."}

    payload = {"origin": {"address": origin}, "destination": {"address": destination},
               "travelMode": "DRIVE", "routingPreference": "TRAFFIC_AWARE", "computeAlternativeRoutes": True}
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json",
               "X-Goog-FieldMask": "routes.duration,routes.distanceMeters,routes.description,routes.warnings,routes.legs.steps.navigationInstruction"}
    try:
        r = HTTP.post("https://routes.googleapis.com/directions/v2:computeRoutes",
                      headers=headers, json=payload, timeout=25)
        r.raise_for_status()
        routes = []
        for route in r.json().get("routes", [])[:3]:
            steps = []
            for leg in route.get("legs", []):
                for step in leg.get("steps", []):
                    instr = (step.get("navigationInstruction") or {}).get("instructions")
                    if instr: steps.append(instr)
            routes.append({"duration": route.get("duration"), "distance_meters": route.get("distanceMeters"),
                           "description": route.get("description"), "warnings": route.get("warnings", []),
                           "steps": steps[:25]})
        return {"status": "success" if routes else "unavailable", "origin": origin,
                "destination": destination, "routes": routes,
                "map_link": "https://www.google.com/maps/dir/?api=1&origin=" + quote(origin) + "&destination=" + quote(destination),
                "critical_warning": "NOT a verified safe evacuation route. Check official evacuation orders and road closures. Do not travel through hazard areas."}
    except requests.RequestException as exc:
        return {"status": "unavailable", "message": "Could not retrieve driving directions.", "error_type": type(exc).__name__}

**Input validation and ADK logging callbacks**

In [7]:
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types

logger = logging.getLogger("readynow")
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
try:
    import google.cloud.logging
    cloud_logging_client = google.cloud.logging.Client(project=PROJECT_ID)
    cloud_logging_client.setup_logging()
    print("Cloud Logging enabled")
except Exception as exc:
    print("Cloud Logging unavailable; using notebook logging:", type(exc).__name__)

COORDS_PATTERN = re.compile(r"(?<!\d)-?\d{1,2}\.\d+\s*,\s*-?\d{1,3}\.\d+")
EMAIL_PATTERN = re.compile(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}")

def redact(value):
    text = str(value or "")
    text = COORDS_PATTERN.sub("[COORDINATES]", text)
    text = EMAIL_PATTERN.sub("[EMAIL]", text)
    return text[:500]

def audit(event_type, agent_name, message=""):
    logger.info(json.dumps({"app": "ReadyNow", "event": event_type, "agent": agent_name,
                            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                            "message_redacted": redact(message)}))

def before_agent_log(callback_context: CallbackContext):
    audit("agent_started", callback_context.agent_name)
    return None

def after_agent_log(callback_context: CallbackContext):
    audit("agent_completed", callback_context.agent_name)
    return None

def _latest_user_message(llm_request: LlmRequest) -> str:
    for content in reversed(llm_request.contents or []):
        if getattr(content, "role", "") == "user":
            return " ".join(p.text for p in (content.parts or []) if getattr(p, "text", None))
    return ""

OFF_TOPIC = re.compile(r"\b(movie review|world cup winner|write (me )?a poem|celebrity gossip|stock tips|video game cheat)\b", re.I)

def before_model_guard(callback_context: CallbackContext, llm_request: LlmRequest):
    message = _latest_user_message(llm_request)
    if message:
        audit("model_input", callback_context.agent_name, message)
    if callback_context.agent_name != "root_agent":
        return None
    if len(message) > 2000:
        return LlmResponse(content=types.Content(role="model", parts=[types.Part(text="Please shorten your emergency preparedness question to 2,000 characters or fewer.")]))
    if message and OFF_TOPIC.search(message) and not re.search(r"\b(emergency|disaster|evacuation|safety|weather|flood|fire|tornado)\b", message, re.I):
        return LlmResponse(content=types.Content(role="model", parts=[types.Part(text="ReadyNow! only assists with disaster preparedness, emergency updates, evacuation information, and safety.")]))
    return None

def after_model_log(callback_context: CallbackContext, llm_response: LlmResponse):
    parts = getattr(getattr(llm_response, "content", None), "parts", None) or []
    message = " ".join(p.text for p in parts if getattr(p, "text", None))
    audit("model_output", callback_context.agent_name, message)
    return None

CALLBACKS = dict(before_agent_callback=before_agent_log,
                 after_agent_callback=after_agent_log,
                 before_model_callback=before_model_guard,
                 after_model_callback=after_model_log)


Cloud Logging enabled


**ADK agents**

In [8]:
import google.adk
import sys

try:
    from google.adk import ContextCacheConfig
    print("Found ContextCacheConfig directly in google.adk")
except ImportError:
    try:
        from google.adk.models import ContextCacheConfig
        print("Found ContextCacheConfig in google.adk.models")
    except ImportError:
        # Let's inspect the module attributes
        import google.adk.models
        print("google.adk.models contents:", dir(google.adk.models))
        import google.adk.runners
        print("google.adk.runners contents:", dir(google.adk.runners))


google.adk.models contents: ['Any', 'BaseLlm', 'LLMRegistry', 'LlmCapabilities', 'LlmRequest', 'LlmResponse', 'ServiceTier', 'TYPE_CHECKING', '_LAZY_PROVIDERS', '_OTHER_LAZY_IMPORTS', '__all__', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_capabilities', '_module', '_name', '_patterns', '_service_tier', '_target_module', 'annotations', 'base_llm', 'base_llm_connection', 'cache_metadata', 'importlib', 'llm_request', 'llm_response', 'registry']
google.adk.runners contents: ['Any', 'AsyncGenerator', 'BaseAgent', 'BaseArtifactService', 'BaseCredentialService', 'BaseMemoryService', 'BaseNode', 'BasePlugin', 'BaseSessionService', 'BaseToolset', 'Callable', 'ContextCacheConfig', 'Event', 'EventActions', 'FINISH_TASK_ERROR_RESULT', 'FINISH_TASK_SUCCESS_RESULT', 'FINISH_TASK_TOOL_NAME', 'Generator', 'GetSessionConfig', 'InMemoryRunner', 'InvocationContext', 'List', 'Literal', 'LiveReques

In [9]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search

weather_agent = Agent(
    name="weather_agent", model=MODEL, description="Checks NWS weather and alerts.",
    instruction="""You are ReadyNow Weather Agent. For a user-provided US city/address, call geocode_location and then get_weather_and_alerts with the returned coordinates. If coordinates are supplied, call get_weather_and_alerts directly. Never guess a location. If the request does not need live weather, state 'Weather check not requested.' If data is unavailable, explicitly say it could not be verified. Include timestamps and official NWS source. Do not confuse no returned alerts with proof of safety.""",
    tools=[geocode_location, get_weather_and_alerts], output_key="weather_report", **CALLBACKS)

search_agent = Agent(
    name="search_agent", model=MODEL, description="Finds official disaster news and emergency instructions.",
    instruction="""You are ReadyNow Emergency Search Agent. Use Google Search to find current official emergency updates relevant to the user's question and location. Prioritize NWS, local emergency management, state DOT, FEMA, fire/police, and then reputable news. Distinguish publication time from event time. Include source URLs and clearly mark anything unverified. Do not claim to have verified road closures or evacuation orders without official evidence. If no current search is needed, give a brief 'No current news required' note.""",
    tools=[google_search], output_key="news_report", **CALLBACKS)

route_agent = Agent(
    name="route_agent", model=MODEL, description="Computes routes via Google Maps Routes API.",
    instruction="""You are ReadyNow Route Agent. Use get_google_maps_route ONLY when user has given both an origin and a destination, or the destination is clearly confirmed by an official source. Never invent a shelter or safe destination. If either is missing, explain what is needed. Google Maps navigation is NOT verified safe evacuation routing. Official evacuation orders, closed roads, and shelter-in-place instructions override maps. If route planning is not requested, state 'Route not requested.'""",
    tools=[get_google_maps_route], output_key="route_report", **CALLBACKS)

answer_agent = Agent(
    name="answer_agent", model=MODEL, description="Answers emergency questions and composes a draft.",
    instruction="""You are ReadyNow Emergency Q&A Agent. Answer the user's emergency question using the reports below when relevant. Weather: {weather_report}
News: {news_report}
Routes: {route_report}
Provide practical protective actions for the specific hazard. Prioritize immediate danger, official instructions, and 911 when appropriate. Distinguish verified facts from general advice. Do not fabricate warnings, closures, shelters, citations, or safe routes. If location is needed but missing, ask for it. Draft an understandable answer with relevant sources and timestamps.""",
    output_key="initial_answer", **CALLBACKS)

validation_agent = Agent(
    name="validation_agent", model=MODEL, description="Validates the draft for safety and clarity.",
    instruction="""Review this draft: {initial_answer}
Check accuracy against available sources, time sensitivity, missing location, uncertainty, clarity, safety instructions, and any unverified evacuation claims. Do not invent new evidence. Return explicit PASS or FAIL with actionable corrections. FAIL any claim that a route is verified safe without authoritative proof, or any fabricated official order. If evidence is insufficient, request cautious language.""",
    output_key="validation_feedback", **CALLBACKS)

refine_agent = Agent(
    name="refine_agent", model=MODEL, description="Produces the final reviewed response.",
    instruction="""Draft: {initial_answer}
Validation: {validation_feedback}
Produce the final concise, easy-to-understand answer. If validation failed on a safety-critical claim, remove that claim and say it is unverified. Use headings 'Current situation', 'What to do', and 'Official sources' when helpful. Clearly state that Google Maps routes are not verified safe evacuation routes. Never invent data or citations. Never hide tool failures.""",
    output_key="final_answer", **CALLBACKS)

emergency_pipeline = SequentialAgent(
    name="emergency_pipeline", description="Weather, news, routes, Q&A, validation and refinement in order.",
    sub_agents=[weather_agent, search_agent, route_agent, answer_agent, validation_agent, refine_agent],
    before_agent_callback=before_agent_log, after_agent_callback=after_agent_log)

greeter_agent = Agent(
    name="greeter_agent", model=MODEL, description="Greets users and explains ReadyNow capabilities.",
    instruction="""Welcome users to ReadyNow!, a proof-of-concept emergency preparedness assistant. Briefly explain that you can check NWS weather alerts, search official disaster updates, calculate Google Maps directions to a specified destination, and explain protective actions. Do not claim to be FEMA or emergency services. Ask what help is needed.""",
    **CALLBACKS)

root_agent = Agent(
    name="root_agent", model=MODEL,
    description="ReadyNow! emergency coordinator: greeting, mission validation, and delegation.",
    instruction="""You are ReadyNow!, an emergency preparedness proof-of-concept assistant. Route greetings or capability questions to greeter_agent. Route disaster, weather, evacuation, and emergency-safety questions to emergency_pipeline, which must complete ALL sequential steps including validation and refinement. Politely refuse unrelated requests; never provide unrelated answers. If someone is in immediate danger, first urge emergency services (911 in the US) and official local instructions. Never guess their location or promise a route is safe. When asked to evacuate, distinguish official orders from ordinary Google Maps navigation.""",
    sub_agents=[greeter_agent, emergency_pipeline], **CALLBACKS)

print("ReadyNow! architecture:", [a.name for a in root_agent.sub_agents])
print("Pipeline:", [a.name for a in emergency_pipeline.sub_agents])


ReadyNow! architecture: ['greeter_agent', 'emergency_pipeline']
Pipeline: ['weather_agent', 'search_agent', 'route_agent', 'answer_agent', 'validation_agent', 'refine_agent']


/tmp/ipykernel_76637/2563239486.py:40: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  emergency_pipeline = SequentialAgent(


**Local testing**

In [10]:
# from google.adk.apps.app import App
# from google.adk.runners import Runner, ContextCacheConfig
# from google.adk.sessions import InMemorySessionService
# import uuid

# # Configure a 5-minute (300 seconds) TTL context cache config
# cache_config = ContextCacheConfig(ttl_seconds=300)

# # Create the App instance incorporating our root agent and context cache config
# app = App(
#     name="readynow_app",
#     root_agent=root_agent,
#     context_cache_config=cache_config
# )

# APP_NAME = "readynow_app"
# USER_ID = "poc_user"
# SESSION_ID = str(uuid.uuid4())

# service = InMemorySessionService()
# await service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)

# # Initialize the Runner with the configured App
# runner = Runner(
#     app=app,
#     session_service=service
# )

# print("Runner successfully initialized with Context-Cached App:", runner)

In [11]:
from google.adk.apps.app import App
from google.adk.runners import Runner, ContextCacheConfig
from google.adk.sessions import InMemorySessionService
import uuid

# Configure a 5-minute (300 seconds) TTL context cache config
cache_config = ContextCacheConfig(ttl_seconds=300)

# Create the App instance incorporating our root agent and context cache config
app = App(
    name="readynow_app",
    root_agent=root_agent,
    context_cache_config=cache_config)

APP_NAME = "readynow_app"
USER_ID = "poc_user"
SESSION_ID = str(uuid.uuid4())
service = InMemorySessionService()
await service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=service)

async def ask_local(prompt):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Enter a nonempty request.")
    responses, errors, authors = [], [], []
    async for event in runner.run_async(
        user_id=USER_ID, session_id=SESSION_ID,
        new_message=types.Content(role="user", parts=[types.Part(text=prompt)])):
        authors.append(event.author)
        if getattr(event, "error_code", None):
            errors.append(str(getattr(event, "error_message", "") or event.error_code))
        if event.is_final_response() and event.content:
            responses.extend(p.text for p in (event.content.parts or []) if getattr(p, "text", None))
    print("PROMPT:", prompt, "\nAGENTS:", authors, "\nRESPONSE:\n", "\n".join(responses), "\nERRORS:", errors)
    assert not errors, f"Agent error: {errors}"
    assert responses, "No final text response. Inspect agent logs."
    return {"response": "\n".join(responses), "authors": authors}

# Start with a simple greeting before testing external APIs.
greeting_result = await ask_local("Hello! What can ReadyNow do?")


{"app": "ReadyNow", "event": "agent_started", "agent": "root_agent", "timestamp_utc": "2026-10-27T21:09:08.509718+00:00", "message_redacted": ""}
INFO:readynow:{"app": "ReadyNow", "event": "agent_started", "agent": "root_agent", "timestamp_utc": "2026-10-27T21:09:08.509718+00:00", "message_redacted": ""}
{"app": "ReadyNow", "event": "model_input", "agent": "root_agent", "timestamp_utc": "2026-10-27T21:09:08.650076+00:00", "message_redacted": "Hello! What can ReadyNow do?"}
INFO:readynow:{"app": "ReadyNow", "event": "model_input", "agent": "root_agent", "timestamp_utc": "2026-10-27T21:09:08.650076+00:00", "message_redacted": "Hello! What can ReadyNow do?"}
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
{"app": "ReadyNow", "event": "model_output", "agent": "root_agent", "timestamp_utc": "2026-10-27T21:09:09.855002+00

PROMPT: Hello! What can ReadyNow do? 
AGENTS: ['root_agent', 'root_agent', 'greeter_agent'] 
RESPONSE:
 Hello! Welcome to ReadyNow!, your proof-of-concept emergency preparedness assistant. I can help you with a few things:

*   **Check NWS weather alerts:** Stay informed about current weather conditions and potential hazards.
*   **Search official disaster updates:** Get the latest information from official sources during an emergency.
*   **Calculate Google Maps directions:** Find the best routes to a specified destination.
*   **Explain protective actions:** Learn what to do before, during, and after an emergency.

Please let me know how I can help you today! 
ERRORS: []


In [14]:
import asyncio

# Execute one at a time with a delay to respect API rate limits (TPM).
print("Starting test 1: Weather check...")
weather_result = await ask_local("Check current weather and active alerts for Denver, Colorado.")

await asyncio.sleep(15)
print("\nStarting test 2: News check...")
news_result = await ask_local("Find current official emergency news for Denver, Colorado.")

await asyncio.sleep(15)
print("\nStarting test 3: Route check...")
route_result = await ask_local("For planning only, give driving directions from Denver, CO to Colorado Springs, CO.")

await asyncio.sleep(15)
print("\nStarting test 4: Safety info...")
safety_result = await ask_local("What should I do during a tornado warning?")

await asyncio.sleep(15)
print("\nStarting test 5: Refusal check...")
refusal_result = await ask_local("Write me a movie review.")
assert "ReadyNow" in refusal_result["response"] or "emergency" in refusal_result["response"].lower()

print("\nPOC test suite finished successfully!")

{"app": "ReadyNow", "event": "agent_started", "agent": "greeter_agent", "timestamp_utc": "2026-10-27T21:15:56.344912+00:00", "message_redacted": ""}
INFO:readynow:{"app": "ReadyNow", "event": "agent_started", "agent": "greeter_agent", "timestamp_utc": "2026-10-27T21:15:56.344912+00:00", "message_redacted": ""}
{"app": "ReadyNow", "event": "model_input", "agent": "greeter_agent", "timestamp_utc": "2026-10-27T21:15:56.354855+00:00", "message_redacted": "Check current weather and active alerts for Denver, Colorado."}
INFO:readynow:{"app": "ReadyNow", "event": "model_input", "agent": "greeter_agent", "timestamp_utc": "2026-10-27T21:15:56.354855+00:00", "message_redacted": "Check current weather and active alerts for Denver, Colorado."}
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.VERTEX_AI, stream: False


Starting test 1: Weather check...


INFO:google_adk.google.adk.models.google_llm:Response received from the model.
{"app": "ReadyNow", "event": "model_output", "agent": "greeter_agent", "timestamp_utc": "2026-10-27T21:15:57.286841+00:00", "message_redacted": ""}
INFO:readynow:{"app": "ReadyNow", "event": "model_output", "agent": "greeter_agent", "timestamp_utc": "2026-10-27T21:15:57.286841+00:00", "message_redacted": ""}
{"app": "ReadyNow", "event": "agent_started", "agent": "emergency_pipeline", "timestamp_utc": "2026-10-27T21:15:57.295543+00:00", "message_redacted": ""}
INFO:readynow:{"app": "ReadyNow", "event": "agent_started", "agent": "emergency_pipeline", "timestamp_utc": "2026-10-27T21:15:57.295543+00:00", "message_redacted": ""}
{"app": "ReadyNow", "event": "agent_started", "agent": "weather_agent", "timestamp_utc": "2026-10-27T21:15:57.297736+00:00", "message_redacted": ""}
INFO:readynow:{"app": "ReadyNow", "event": "agent_started", "agent": "weather_agent", "timestamp_utc": "2026-10-27T21:15:57.297736+00:00", "

_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Quota exceeded for aiplatform.googleapis.com/global_generate_content_input_tokens_per_minute_per_base_model with base model: gemini-2.5-flash-ga. Please submit a quota increase request. https://cloud.google.com/vertex-ai/docs/generative-ai/quotas-genai.', 'status': 'RESOURCE_EXHAUSTED'}}

In [ ]:
# async def ask_cached_local(prompt):
#     if not isinstance(prompt, str) or not prompt.strip():
#         raise ValueError("Enter a nonempty request.")
#     responses, errors, authors = [], [], []
#     async for event in runner.run_async(
#         user_id=USER_ID, session_id=SESSION_ID,
#         new_message=types.Content(role="user", parts=[types.Part(text=prompt)])):
#         authors.append(event.author)
#         if getattr(event, "error_code", None):
#             errors.append(str(getattr(event, "error_message", "") or event.error_code))
#         if event.is_final_response() and event.content:
#             responses.extend(p.text for p in (event.content.parts or []) if getattr(p, "text", None))
#     print("PROMPT:", prompt, "\nAGENTS:", authors, "\nRESPONSE:\n", "\n".join(responses), "\nERRORS:", errors)
#     assert not errors, f"Agent error: {errors}"
#     assert responses, "No final text response."
#     return {"response": "\n".join(responses), "authors": authors}

# # Test greeting
# greeting_test = await ask_cached_local("Hello! What can ReadyNow do?")